In [82]:
import requests
import csv
import re
from bs4 import BeautifulSoup
from urllib.parse import urljoin

In [83]:
BASE = 'http://en.wikipedia.org'
OUT_PATH = 'nh_4000_footers.csv'

In [ ]:
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "mtn-grid/0.1 (contact: shay.subramanian@gmail.com)",
    "From": "shay.subramanian@gmail.com",
})

In [85]:
URL = 'https://en.wikipedia.org/wiki/Four-thousand_footers'
response = SESSION.get(URL)
html_content = response.content

# Create the BeautifulSoup object
soup = BeautifulSoup(html_content, 'html.parser')

# Get the full HTML as a string
full_html_string = str(soup)

# Get the full HTML with nice indentation
prettified_html = soup.prettify()

print(prettified_html)

<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled vector-toc-available" dir="ltr" lang="en">
 <head>
  <meta charset="utf-8"/>
  <title>
   Four-thousand footers - Wikipedia
  </title>
  <script>
   (function(){var className="client-js vector-feature-language-in-header-enabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-cust

In [86]:
# 1) Find the NH heading (the <h2 id="The_New_Hampshire_list">)
nh_heading = soup.find(id="The_New_Hampshire_list")

# 2) The section wrapper div is the *parent* (div.mw-heading ...)
nh_root = nh_heading.parent

# Checkpoint: print what you have
print("nh_heading tag:", nh_heading.name)
print("nh_root tag:", nh_root.name, "class:", nh_root.get("class"))

nh_heading tag: h2
nh_root tag: div class: ['mw-heading', 'mw-heading2']


In [87]:
# Find the NEXT div with class "div-col" after nh_root
divcol = nh_root.find_next(
    "div",
    class_="div-col"
)

# Checkpoint
print(divcol.name, divcol.get("class"))

div ['div-col']


In [88]:
ol = divcol.find("ol")
lis = ol.find_all("li")

print("Number of list items:", len(lis))
print("First li preview:", lis[0].get_text(" ", strip=True)[:120])

Number of list items: 48
First li preview: Washington :	6,288 ft (1,917 m) AT*


In [89]:
names = []

for li in lis:
    a = li.find("a")
    if not a:
        continue
    name = a.get_text(strip=True)
    names.append(name)

# Checkpoint
print("Extracted names:", len(names))
print("First 10 names:", names[:10])

Extracted names: 48
First 10 names: ['Washington', 'Adams', 'Jefferson', 'Monroe', 'Madison', 'Lafayette', 'Lincoln', 'South Twin', 'Carter Dome', 'Moosilauke']


In [90]:
links = []

for li in lis:
    a = li.find("a", href=True)
    if not a:
        continue
    href = a["href"]
    links.append(href)

print("First 5 hrefs:", links[:5])

First 5 hrefs: ['/wiki/Mount_Washington_(New_Hampshire)', '/wiki/Mount_Adams_(New_Hampshire)', '/wiki/Mount_Jefferson_(New_Hampshire)', '/wiki/Mount_Monroe_(New_Hampshire)', '/wiki/Mount_Madison']


In [ ]:
def get_peak_name_from_page(peak_url):
    """
    Input: full Wikipedia URL to a peak page
    Output: cleaned peak name from <h1 id="firstHeading">
    """
    response = SESSION.get(peak_url)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    heading = soup.find("h1", id="firstHeading")
    if heading is None:
        raise ValueError(f"No <h1 id='firstHeading'> found on {peak_url}")

    name = heading.get_text(strip=True)

    # Remove trailing " (New Hampshire)" if present
    if name.endswith(" (New Hampshire)"):
        name = name.replace(" (New Hampshire)", "")

    return name


In [92]:
def make_peak_id(lat, lon):
    lat_str = f"{lat:.5f}"
    lon_str = f"{lon:.5f}"

    peak_id = re.sub(r"[^0-9]", "", lat_str + lon_str)

    return peak_id, float(lat_str), float(lon_str)

In [93]:
def get_decimal_coords_from_page(peak_url):
    """
    Input: full Wikipedia URL to a peak page
    Output: (latitude, longitude) as floats in decimal degrees
    """
    response = SESSION.get(peak_url)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    # Preferred: Wikipedia geo microformat (decimal lat;lon)
    geo = soup.find("span", class_="geo")
    if geo is None:
        raise ValueError(f"No geo coordinates found on {peak_url}")

    try:
        lat_str, lon_str = geo.get_text(strip=True).split(";")
        lat = float(lat_str)
        lon = float(lon_str)
    except Exception:
        raise ValueError(f"Could not parse coordinates on {peak_url}: {geo.get_text()}")

    return lat, lon

In [ ]:
rows = []

for href in links:
    peak_url = urljoin(BASE, href)

    peak_name = get_peak_name_from_page(peak_url)
    lat_raw, lon_raw = get_decimal_coords_from_page(peak_url)

    peak_id, lat, lon = make_peak_id(lat_raw, lon_raw)

    row = {
        "peak_id": peak_id,
        "peak_name": peak_name,
        "state": "New Hampshire",
        "latitude": lat,
        "longitude": lon,
        "enter_m": 80,
        "exit_m": 120,
        "exit_consec_points": 5,
    }
    rows.append(row)

print("Rows collected:", len(rows))
print("Sample row:", rows[0])
rows

Rows collected: 48
Sample row: {'peak_id': '44270507130325', 'peak_name': 'Mount Washington', 'state': 'New Hampshire', 'latitude': 44.2705, 'longitude': -71.30325, 'enter_m': 80, 'exit_m': 120, 'exit_consec_points': 5}


[{'peak_id': '44270507130325',
  'peak_name': 'Mount Washington',
  'state': 'New Hampshire',
  'latitude': 44.2705,
  'longitude': -71.30325,
  'enter_m': 80,
  'exit_m': 120,
  'exit_consec_points': 5},
 {'peak_id': '44320567129139',
  'peak_name': 'Mount Adams (New Hampshire)',
  'state': 'New Hampshire',
  'latitude': 44.32056,
  'longitude': -71.29139,
  'enter_m': 80,
  'exit_m': 120,
  'exit_consec_points': 5},
 {'peak_id': '44304207131685',
  'peak_name': 'Mount Jefferson (New Hampshire)',
  'state': 'New Hampshire',
  'latitude': 44.3042,
  'longitude': -71.31685,
  'enter_m': 80,
  'exit_m': 120,
  'exit_consec_points': 5},
 {'peak_id': '44255567132250',
  'peak_name': 'Mount Monroe',
  'state': 'New Hampshire',
  'latitude': 44.25556,
  'longitude': -71.3225,
  'enter_m': 80,
  'exit_m': 120,
  'exit_consec_points': 5},
 {'peak_id': '44328827127678',
  'peak_name': 'Mount Madison',
  'state': 'New Hampshire',
  'latitude': 44.32882,
  'longitude': -71.27678,
  'enter_m': 80,